# California Wildfires and Pollution (PM2.5) on California Prison and Detention Facilities

Outline: 
- Fire Vectors: (Wildfire Perimeter GeoPackage Data + NASA FIRMS) # The Cause
- Geography: (OpenTopography Elevation) # The Context
- Pollution Data: (AirNow + PurpleAir) # The Result
- Satellite Atmosphere: (GEE + NASA TROPOMI) # Enhanced Data

The CALFIRE Wildfire Perimeter GeoPackage: 
https://gis.data.ca.gov/datasets/CALFIRE-Forestry::california-fire-perimeters-all/explore

Annual AQI conc by monitor data: 
aqs.epa.gov/aqsweb/airdata/download_files.html

In [54]:
import os
import re
import ee
import json
import datetime 
import requests
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt

import folium as fm
import geopandas as gpd

from datetime import datetime
from dotenv import load_dotenv, find_dotenv
from sklearn import linear_model as lm

from geopy.geocoders import Nominatim
from geopy.geocoders import GoogleV3
from geopy.extra.rate_limiter import RateLimiter

In [2]:
ee.Authenticate()
ee.Initialize(project='cawfppproject')

In [3]:
# Loading environment variables in .env

load_dotenv()

PURPLEAIR_API_KEY = os.getenv("PURPLEAIR_API_KEY")

In [4]:
# Obtaining Path files to the data in data folder

directory_name = os.getcwd()
directory_path = Path(directory_name)

# Loading consolidated facilities dataframe: 
final_facilities_data = pd.read_pickle(directory_path.joinpath('data/final_facilities_data.pkl'))

# CA facilities geometries geopackage file path:
ca_facilities_geo_path = directory_path.joinpath("data/final_facilities_polygons.gpkg")

# CA wildfire perimeters geopackage file path:
ca_wildfire_perim_path = directory_path.joinpath("data/California_Historic_Fire_Perimeters_586217350401785615.gpkg")

# NASA FIRMS archived data path: 
nasa_firms_path = directory_path.joinpath("data/fire_archive_SV-C2_705647.csv")

# NASA FIRMS DEM data path:
nasa_firms_dem_path = directory_path.joinpath("data/NASA_Fire_Elevations_Final.csv")

# EPA daily PM2.5 path: 
epa_daily_pm_summaries_path = directory_path.joinpath("data/epa_pm_daily_summaries_2020_2025")

## Feature Engineering Wildfire Data

### California Wildfire Perimeter GeoPackage and Combining with Facilities DataFrame

In [5]:
# Loading in the California Wildfire Perimeter Geopackage with Geopandas. 
wildfire_perim_gdf = gpd.read_file(ca_wildfire_perim_path)

# Filtering for dates from 2020 - 2025.
wildfire_perim_gdf = wildfire_perim_gdf[wildfire_perim_gdf['YEAR_'] >= 2020.0]

# Converting dates.
wildfire_perim_gdf['ALARM_DATE'] = pd.to_datetime(wildfire_perim_gdf['ALARM_DATE'], errors='coerce')
wildfire_perim_gdf['CONT_DATE'] = pd.to_datetime(wildfire_perim_gdf['CONT_DATE'], errors='coerce')

# Handling missing end dates.
wildfire_perim_gdf['CONT_DATE'] = wildfire_perim_gdf['CONT_DATE'].fillna(wildfire_perim_gdf['ALARM_DATE'] + pd.Timedelta(days=30))

# Setting EPSG: 3310 for calculations
wildfire_perim_gdf = wildfire_perim_gdf.to_crs(3310)
wildfire_perim_gdf

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: Non-conformant content for record 1 in column ALARM_DATE, 2025-01-07T08:00:00.0Z, successfully parsed
  return ogr_read(


,YEAR_,STATE,AGENCY,UNIT_ID,FIRE_NAME,INC_NUM,ALARM_DATE,CONT_DATE,CAUSE,C_METHOD,OBJECTIVE,GIS_ACRES,COMMENTS,COMPLEX_NAME,IRWINID,FIRE_NUM,COMPLEX_ID,DECADES,geometry
0,2025.0,CA,CDF,LDF,PALISADES,00000738,2025-01-07 08:00:00+00:00,2025-01-31 08:00:00+00:00,14,7.0,1.0,23448.882812,None,None,{A7EA5D21-F882-44B8-BF64-44AB11059DC1},None,None,2020-January 2025,"MULTIPOLYGON (((131998.056 -435368.234, 131998..."
1,2025.0,CA,CDF,LAC,EATON,00009087,2025-01-08 08:00:00+00:00,2025-01-31 08:00:00+00:00,14,7.0,1.0,14056.260742,None,None,{72660ADC-B5EF-4D96-A33F-B4EA3740A4E3},None,None,2020-January 2025,"MULTIPOLYGON (((176706.656 -418166.172, 176705..."
2,2025.0,CA,CDF,ANF,HUGHES,00250270,2025-01-22 08:00:00+00:00,2025-01-28 08:00:00+00:00,14,7.0,1.0,10396.798828,None,None,{994072D2-E154-434A-BB95-6F6C94C40829},None,None,2020-January 2025,"MULTIPOLYGON (((132176.399 -380697.134, 132154..."
3,2025.0,CA,CCO,VNC,KENNETH,00003155,2025-01-09 08:00:00+00:00,2025-02-04 08:00:00+00:00,14,2.0,1.0,998.737793,from OES Intel 24,None,{842FB37B-7AC8-4700-BB9C-028BF753D149},None,None,2020-January 2025,"MULTIPOLYGON (((121966.746 -426575.295, 121943..."
4,2025.0,CA,CDF,LDF,HURST,00003294,2025-01-07 08:00:00+00:00,2025-01-09 08:00:00+00:00,14,7.0,1.0,831.385498,None,None,{F4E810AD-CDF3-4ED4-B63F-03D43785BA7B},None,None,2020-January 2025,"MULTIPOLYGON (((140773.831 -408331.971, 140774..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2031,2020.0,CA,CCO,KRN,OUTLET,02029995,2020-07-16 07:00:00+00:00,2020-07-16 07:00:00+00:00,14,1.0,1.0,1.066872,SRA,None,,None,None,2020-January 2025,"MULTIPOLYGON (((95523.839 -337830.691, 95521.8..."
2032,2020.0,CA,CDF,SLU,PRICE,00006674,2020-05-31 07:00:00+00:00,2020-05-31 07:00:00+00:00,10,6.0,1.0,1.057545,None,None,{347EF255-2E5D-403C-A19B-FA251D7948A2},None,None,2020-January 2025,"MULTIPOLYGON (((-58503.584 -315600.447, -58507..."
2033,2020.0,CA,CDF,LNU,TWIN,00019608,2020-11-23 08:00:00+00:00,2020-11-23 08:00:00+00:00,5,6.0,1.0,1.042918,Jason Freyer,None,{D3A285DF-3FC4-45B8-A1D6-D71F66C0A9B2},None,None,2020-January 2025,"MULTIPOLYGON (((-186419.031 33556.726, -186418..."
2034,2020.0,CA,CDF,SLU,COUNTRY,00012211,2020-09-17 07:00:00+00:00,2020-09-17 07:00:00+00:00,8,6.0,1.0,1.016899,None,None,{10ACE5F0-A540-444A-BE7E-199D2B22FD17},None,None,2020-January 2025,"MULTIPOLYGON (((-50629.53 -330217.066, -50629...."


In [6]:
# Want to find features that encompass: facilities distance from wildfires, intensities of each wildfire, 
# magnitude of the biggest fire and any intersecting fires that have occurred near facilities.

# Setting EPSG: 3310 for calculations
final_facilities_data = final_facilities_data.set_geometry("Geometry")
final_facilities_data = final_facilities_data.to_crs(3310)

final_facilities_data['Date'] = pd.to_datetime(final_facilities_data['Date'], errors='coerce')
final_facilities_data['Date'] = final_facilities_data['Date'].dt.strftime("%Y-%m")

wildfire_perim_gdf["year_month"] = wildfire_perim_gdf['ALARM_DATE'].dt.strftime("%Y-%m")

# Merge at Year-Month to find distance between facilities and wildfires
merged_wildfire_fac = final_facilities_data.merge(wildfire_perim_gdf, left_on='Date', right_on='year_month')
merged_wildfire_fac['Distance'] = merged_wildfire_fac['Geometry'].distance(merged_wildfire_fac['geometry'])

# Creating distance in km and an intensity score column that sets intensity score = GIS ACRES / (DISTANCE + 1)**2
merged_wildfire_fac['distance_km'] = merged_wildfire_fac['Distance'] / 1000
merged_wildfire_fac['fire_magnitude_score'] = (
    (merged_wildfire_fac['GIS_ACRES'])/(merged_wildfire_fac['distance_km'] + 1)**2
)

# Filtering for distance less than or equal to 100km to create three separate zones and aggregating data
bins = [-1, 10, 50, 100]
bin_labels = ['fires_at_10km', 'fires_at_50km', 'fires_at_100km']

filtered_100km_wildfires = merged_wildfire_fac[merged_wildfire_fac['distance_km'] <= 100].copy()
filtered_100km_wildfires['has_intersecting_fires'] = (filtered_100km_wildfires['distance_km'] <= .10).astype(int)

filtered_100km_wildfires['distance_zones'] = pd.cut(x=filtered_100km_wildfires['distance_km'], bins=bins, labels=bin_labels)

distance_zones_pivot = pd.pivot_table(
    filtered_100km_wildfires, 
    values='FIRE_NAME', 
    index=['ID', 'year_month'], 
    columns='distance_zones', 
    aggfunc='count', 
    fill_value=0,
    observed=False
).add_prefix('number_of_')

aggregated_wildfire_data = filtered_100km_wildfires.groupby(['ID', 'year_month']).agg(
    min_distance_km=('distance_km', 'min'),
    max_magnitude_score=('fire_magnitude_score', 'max'),
    total_magnitude=('fire_magnitude_score', 'sum'),
    max_acres=('GIS_ACRES', 'max'),
    total_acres=('GIS_ACRES', 'sum'),
    intersecting_fires=('has_intersecting_fires', 'max')
)

wildfire_features_df = aggregated_wildfire_data.merge(distance_zones_pivot, left_on=['ID', 'year_month'], right_on=['ID', 'year_month']).reset_index()
wildfire_features_df = wildfire_features_df.rename(columns={'year_month': 'Date'})
wildfire_features_df

,ID,Date,min_distance_km,max_magnitude_score,total_magnitude,max_acres,total_acres,intersecting_fires,number_of_fires_at_10km,number_of_fires_at_50km,number_of_fires_at_100km
0,0,2020-07,83.848412,18.601251,18.614176,143836.406250,143929.453125,0,0,0,2
1,0,2020-08,98.816364,0.013710,0.013710,136.592819,136.592819,0,0,0,1
2,0,2020-09,31.691595,147.304114,149.022120,157429.859375,166339.453125,0,0,1,2
3,0,2021-03,91.145479,0.001250,0.001250,10.612824,10.612824,0,0,0,1
4,0,2021-04,72.123750,0.008417,0.008417,45.007931,45.007931,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
2696,78,2024-06,65.802611,0.007254,0.007254,32.373474,32.373474,0,0,0,1
2697,78,2024-07,71.782568,0.289942,0.291739,1596.656494,1606.172119,0,0,0,2
2698,78,2024-09,90.488849,0.004000,0.004000,33.479469,33.479469,0,0,0,1
2699,78,2024-10,92.724527,0.046829,0.055466,411.360443,489.978394,0,0,0,2


### NASA FIRMS (Fire Information for Resource Management Systems) Data and Combining with Facilities DataFrame

In [7]:
# NASA FIRMS archived VIIRS(S-NPP) data contains daily information about thermal energy released from active fires from 2020-2025,
# which allows us to understand the intensity of the wildfires that occurred during this timeline.

# Want to specifically find frp (fire radiative power) of wildfires that occurred closer to the facilities, night
# detection of strong wildfires, and duration of fires to have a descriptive understanding of wildfire impact on facilities.

# Creating a GeoDataFrame of NASA FIRMS data
nasa_firms_df = pd.read_csv(nasa_firms_path)
nasa_firms_gdf = gpd.GeoDataFrame(
    nasa_firms_df,
    geometry=gpd.points_from_xy(nasa_firms_df.longitude, nasa_firms_df.latitude),
    crs="EPSG:4326"          
)
nasa_firms_gdf = nasa_firms_gdf.to_crs(3310)

nasa_firms_gdf['acq_date'] = pd.to_datetime(nasa_firms_gdf['acq_date'], errors='coerce')
nasa_firms_gdf['year_month'] = nasa_firms_gdf['acq_date'].dt.strftime("%Y-%m")

In [8]:
# Creating a buffer at 50km from the facilities dataframe to join with NASA FIRMS data
fac_data_buffer = final_facilities_data.copy()
fac_data_buffer['buffer'] = fac_data_buffer.buffer(50000)
fac_data_buffer = fac_data_buffer.set_geometry('buffer').to_crs(3310)

unique_months = sorted(set(nasa_firms_gdf['year_month']).intersection(set(fac_data_buffer['Date'])))
sorted_fac_firms_list = []
for month in unique_months: 
    sorted_month_firms_set = nasa_firms_gdf[nasa_firms_gdf['year_month'] == month]
    sorted_month_fac_set = fac_data_buffer[fac_data_buffer['Date'] == month]

    sorted_firms_fac_join_df = gpd.sjoin(
        sorted_month_firms_set,
        sorted_month_fac_set[['ID', 'Date', 'Geometry', 'buffer']],
        how='inner',
        predicate='within'
    )

    sorted_fac_firms_list.append(sorted_firms_fac_join_df)

fac_firms_df = pd.concat(sorted_fac_firms_list)

# Filtering for FRP > 10 and distance from facilities to do calculations and aggregations
fac_firms_filtered = fac_firms_df[fac_firms_df['frp'] > 10].copy()

fac_firms_filtered['Distance'] = fac_firms_filtered['Geometry'].distance(fac_firms_filtered['geometry'])
fac_firms_filtered['distance_km'] = fac_firms_filtered['Distance'] / 1000

fac_firms_filtered['burn_intensity_score'] = (
    (fac_firms_filtered['frp']) / (fac_firms_filtered['distance_km'] + 1) ** 2 
)

aggregated_firms_features = fac_firms_filtered.groupby(['ID', 'year_month']).agg(
    max_frp=('frp', 'max'),
    total_frp=('frp', 'sum'),
    total_thermal_score=('burn_intensity_score', 'sum'),
    fire_day_count=('acq_date', 'nunique'),
    pixel_count=('frp', 'count'),
    night_detections=('daynight', lambda x: (x == 'N').sum())
).add_prefix('firms_50km_').reset_index()

aggregated_firms_features['firms_50km_night_fire_ratio'] = (aggregated_firms_features['firms_50km_night_detections']) / (aggregated_firms_features['firms_50km_pixel_count']).fillna(0) 
aggregated_firms_features = aggregated_firms_features.rename(columns={'year_month': 'Date'})
aggregated_firms_features

,ID,Date,firms_50km_max_frp,firms_50km_total_frp,firms_50km_total_thermal_score,firms_50km_fire_day_count,firms_50km_pixel_count,firms_50km_night_detections,firms_50km_night_fire_ratio
0,0,2020-09,562.70,20558.99,9.712470,11,442,117,0.264706
1,0,2020-10,61.21,169.66,0.084065,4,6,0,0.000000
2,0,2020-11,17.93,57.67,0.056736,1,4,0,0.000000
3,0,2021-12,10.31,10.31,0.007488,1,1,0,0.000000
4,0,2022-02,14.58,14.58,0.193622,1,1,0,0.000000
...,...,...,...,...,...,...,...,...,...
3249,78,2024-09,42.23,430.81,0.343331,11,26,1,0.038462
3250,78,2024-10,35.68,91.89,0.061128,4,5,0,0.000000
3251,78,2024-11,10.98,10.98,0.005381,1,1,0,0.000000
3252,78,2024-12,52.27,370.18,0.286884,6,11,0,0.000000


In [9]:
# Final wildfire features merged with facility data
facilities_df = final_facilities_data.copy()
wildfire_package_facilities_merge = facilities_df.merge(wildfire_features_df, how='left', on=['ID', 'Date'])
final_wildfire_features = wildfire_package_facilities_merge.merge(aggregated_firms_features, how='left', on=['ID', 'Date']).fillna(0)

final_wildfire_features.drop(columns=['Location', 'Address', 'Perimeter', 'ID', 'Latitude', 'Longitude', 'Centroid', 'Geometry', 'firms_50km_night_detections'], inplace = True)
final_wildfire_features

,Institution Name,Population,Date,Percent of Total Population,Facility Type,Area,min_distance_km,max_magnitude_score,total_magnitude,max_acres,...,intersecting_fires,number_of_fires_at_10km,number_of_fires_at_50km,number_of_fires_at_100km,firms_50km_max_frp,firms_50km_total_frp,firms_50km_total_thermal_score,firms_50km_fire_day_count,firms_50km_pixel_count,firms_50km_night_fire_ratio
0,Avenal State Prison (ASP),4230.0,2024-09,4.657616,Adult Facility,409659.422287,20.445998,11.512470,12.723598,17614.501953,...,0.0,0.0,3.0,10.0,135.46,7539.60,3.435115,4.0,167.0,0.08982
1,Avenal State Prison (ASP),4431.0,2024-08,4.882322,Adult Facility,409659.422287,47.761541,0.013578,0.027881,93.720016,...,0.0,0.0,1.0,6.0,0.00,0.00,0.000000,0.0,0.0,0.00000
2,Avenal State Prison (ASP),4056.0,2024-10,4.473858,Adult Facility,409659.422287,58.614185,0.002694,0.009234,14.273177,...,0.0,0.0,0.0,6.0,130.86,782.16,88.895563,1.0,10.0,0.00000
3,Avenal State Prison (ASP),4717.0,2024-01,5.103266,Adult Facility,409659.422287,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.00,0.00,0.000000,0.0,0.0,0.00000
4,Avenal State Prison (ASP),4796.0,2024-03,5.214631,Adult Facility,409659.422287,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,33.43,33.43,0.023573,1.0,1.0,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5252,Otay Mesa Detention Center,1241.0,2024-09,46.971991,ICE Detention,26231.019040,52.603041,0.084247,0.098347,856.154236,...,0.0,0.0,0.0,3.0,166.44,5160.94,4.053906,6.0,110.0,0.00000
5253,Otay Mesa Detention Center,1320.0,2024-10,44.639838,ICE Detention,26231.019040,47.047573,0.167037,0.201092,411.360443,...,0.0,0.0,2.0,0.0,76.25,524.46,0.311930,6.0,20.0,0.00000
5254,Otay Mesa Detention Center,1364.0,2024-11,45.603477,ICE Detention,26231.019040,1.493342,11.075086,11.122651,68.851067,...,0.0,1.0,1.0,2.0,66.76,431.95,0.358535,4.0,12.0,0.00000
5255,Otay Mesa Detention Center,1373.0,2024-12,45.873705,ICE Detention,26231.019040,20.013669,0.050034,0.050034,22.093618,...,0.0,0.0,1.0,0.0,47.55,248.06,0.151767,2.0,8.0,0.50000


## Feature Engineering Topography Data

### OpenTopography Data Combined with Facilities Data

In [10]:
# Turning facility points, and buffer points into FeatureCollections
fac_4326 = final_facilities_data.copy().to_crs(4326)
fac_4326 = fac_4326.drop_duplicates(subset=['Institution Name'], keep='first')

fac_features = []
for index, row in fac_4326.iterrows():
    lon = row['Centroid'].x
    lat = row['Centroid'].y
    
    geom = ee.Geometry.Point([lon, lat])
    
    feature = ee.Feature(geom, {'ID': row['ID']})
    fac_features.append(feature)
    
facility_points_fc = ee.FeatureCollection(fac_features)
# Buffer created to see the surrounding elevation around facilities to determine if it sits in a valley or not.
facility_buffers_fc = facility_points_fc.map(lambda f: f.buffer(5000))

In [11]:
# Obtaining elevation data for facility points, and buffer points
dem = ee.ImageCollection('COPERNICUS/DEM/GLO30').select('DEM').mosaic()

fac_elevation_dem_data = dem.reduceRegions(collection=facility_points_fc, reducer=ee.Reducer.mean(), scale=30)
buffer_elevation_dem_data = dem.reduceRegions(collection=facility_buffers_fc, reducer=ee.Reducer.mean(), scale=30)

fac_ela_all_features = fac_elevation_dem_data.getInfo()['features']
buffer_ela_all_features = buffer_elevation_dem_data.getInfo()['features']

fac_buffer_elevation = pd.DataFrame([
    {
        'ID': f['properties']['ID'],
        'facility_elevation': f['properties']['mean'],
        '5km_buffer_elevation': b['properties']['mean']
    } 
    for f, b in zip(fac_ela_all_features, buffer_ela_all_features)
])

fac_buffer_elevation['valley_index'] = fac_buffer_elevation['facility_elevation'] - fac_buffer_elevation['5km_buffer_elevation']
fac_buffer_elevation

,ID,facility_elevation,5km_buffer_elevation,valley_index
0,37,234.187286,261.699993,-27.512706
1,58,783.003296,779.601931,3.401365
2,12,1253.286133,1293.518672,-40.232539
3,49,1190.277100,1335.470494,-145.193394
4,25,14.136493,12.260484,1.876009
...,...,...,...,...
74,60,897.530212,900.511744,-2.981532
75,44,113.793739,113.325688,0.468052
76,78,12.974571,7.688057,5.286514
77,48,128.655075,133.284905,-4.629830


In [12]:
# FIRMS data utilized fac_firms_filtered data points, and elevation dem data was sent via Google Drive
# Consolidated all elevation data into one.
firms_elevation_dem_data = pd.read_csv(nasa_firms_dem_path)

def extract_coordinates(geo_str):
    geo_json = json.loads(geo_str)
    return geo_json['coordinates']

# Extracting latitude and longitude from DEM data
firms_elevation_dem_data['coordinates'] = firms_elevation_dem_data['.geo'].apply(extract_coordinates)
firms_elevation_dem_data['longitude'] = firms_elevation_dem_data['coordinates'].apply(lambda x: x[0])
firms_elevation_dem_data['latitude'] = firms_elevation_dem_data['coordinates'].apply(lambda x: x[1])
firms_dem = firms_elevation_dem_data[['mean', 'latitude', 'longitude']].copy()
firms_dem = firms_dem.rename(columns={'mean': 'elevation_mean'})

# Copying fac_firms_filtered and rounding to allow for safe merging between fac_firms_filtered and DEM data found
og_fac_firms = fac_firms_filtered.copy()
og_fac_firms['latitude'] = og_fac_firms['latitude'].round(5)
og_fac_firms['longitude'] = og_fac_firms['longitude'].round(5)

firms_dem['latitude'] = firms_dem['latitude'].round(5)
firms_dem['longitude'] = firms_dem ['longitude'].round(5)

fac_firms_elevated = og_fac_firms.merge(
    firms_dem, 
    on=['latitude', 'longitude'], 
    how='left'
)

# Aggregating to get aggregated elevation data
agg_firms_elevated = fac_firms_elevated.groupby(['ID', 'Date']).agg(
    fire_50km_elevation_mean = ('elevation_mean', 'mean'),
    fire_50km_elevation_max = ('elevation_mean', 'max')
).reset_index()

# Combining with facility and buffer elevation
final_elevation_features = agg_firms_elevated.merge(
    fac_buffer_elevation,
    on='ID',
    how='left'
)

final_elevation_features

,ID,Date,fire_50km_elevation_mean,fire_50km_elevation_max,facility_elevation,5km_buffer_elevation,valley_index
0,0,2020-09,1068.741063,1924.012573,29.472229,59.493360,-30.021131
1,0,2020-10,1124.144653,1336.420898,29.472229,59.493360,-30.021131
2,0,2020-11,293.506351,398.442047,29.472229,59.493360,-30.021131
3,0,2021-12,478.716858,478.716858,29.472229,59.493360,-30.021131
4,0,2022-02,545.715820,545.715820,29.472229,59.493360,-30.021131
...,...,...,...,...,...,...,...
3249,78,2024-09,16.387347,29.846750,12.974571,7.688057,5.286514
3250,78,2024-10,16.031475,29.079716,12.974571,7.688057,5.286514
3251,78,2024-11,25.609886,25.609886,12.974571,7.688057,5.286514
3252,78,2024-12,33.661182,42.823826,12.974571,7.688057,5.286514


## Feature Engineering Pollution Data

### EPA Daily PM2.5 Data

In [151]:
# Concatenating EPA Daily PM2.5 Summaries from 2020-2025 into one dataframe.

epa_summaries_list = []
def extract_epa_pm_data(path):
    for i in range(6):
        epa_yearly_summary_file = epa_daily_pm_summaries_path.joinpath(f"daily_pm_202{i}_data.csv")
        epa_yearly_df = pd.read_csv(epa_yearly_summary_file)

        if not epa_yearly_df.empty:
            epa_summaries_list.append(epa_yearly_df)
        else: 
            print(f"Could not find EPA Yearly Summary File for Year: 202{i}")
    result = pd.concat(epa_summaries_list)
    return result

epa_daily_pm_california = extract_epa_pm_data(epa_daily_pm_summaries_path)
epa_cal_filtered = epa_daily_pm_california[['Date', 'Site ID', 'POC', 'Daily Mean PM2.5 Concentration', 
                                            'Daily AQI Value', 'Site Latitude', 'Site Longitude']] 

# Group data to collapse POCs
epa_cal_grouped = epa_cal_filtered.groupby(['Site ID', 'Date']).agg({
    'Daily Mean PM2.5 Concentration': 'mean',
    'Daily AQI Value': 'max',
    'Site Latitude': 'first',                
    'Site Longitude': 'first' 
}).reset_index()

# Utilize grouped data to find distance from facilities
epa_fac = final_facilities_data.copy()

# Converting epa df into a GeoDataFrame
epa_cal_grouped = epa_cal_grouped.rename(columns={
    'Site Latitude': 'Latitude',
    'Site Longitude': 'Longitude'
})

epa_cal_grouped_gdf = gpd.GeoDataFrame(
    epa_cal_grouped, geometry=gpd.points_from_xy(epa_cal_grouped.Longitude, epa_cal_grouped.Latitude), crs="EPSG:4326"
) 
epa_cal_grouped_gdf = epa_cal_grouped_gdf.to_crs(epsg=3310)

# Simplifying and dropping duplicates for facilities and EPA Monitor Sites
epa_single_fac = epa_fac.drop_duplicates(subset=['Institution Name', 'ID'])
epa_single_cal = epa_cal_grouped_gdf.drop_duplicates(subset=['Site ID'])

merged_cross_epa_fac = pd.merge(epa_single_fac, epa_single_cal, how='cross', suffixes=('_fac', '_epa'))
merged_cross_epa_fac['distance_fac_epa'] = merged_cross_epa_fac['Geometry'].distance(merged_cross_epa_fac['geometry'])

# Filter for monitor distance that is < 50km away from facilities
merged_distance_epa_fac = merged_cross_epa_fac[merged_cross_epa_fac['distance_fac_epa'] <=  50000]

# Noting missing facilities without any monitors 50km away from them
original_fac_ids = set(epa_fac['ID'].unique())
epa_ids = set(merged_distance_epa_fac['ID'].unique())
missing_ids = original_fac_ids - epa_ids

# Merge back with De-duplicated EPA data as that contains information from the whole timeseries (2020-2025)
spatial_lookup = merged_distance_epa_fac[['ID', 'Institution Name', 'Site ID', 'distance_fac_epa']]

epa_fac_timeseries = pd.merge(
    spatial_lookup, 
    epa_cal_grouped, 
    on='Site ID', 
    how='inner'
)

# Aggregate EPA PM2.5 and facilities data
epa_fac_agg_df = epa_fac_timeseries.groupby(['Date', 'ID', 'Institution Name']).agg(
    max_pm_conc=('Daily Mean PM2.5 Concentration','max'),
    max_aqi=('Daily AQI Value','max'),
    min_fac_monitor_dist=('distance_fac_epa','min'),
    monitor_count=('Site ID', 'count')
).reset_index()

epa_fac_agg_df = epa_fac_agg_df[epa_fac_agg_df['Date'] <= '09/01/2025']
epa_fac_agg_df

,Date,ID,Institution Name,max_pm_conc,max_aqi,min_fac_monitor_dist,monitor_count
0,01/01/2020,0,Pelican Bay State Prison (PBSP),8.6,48,11177.073824,2
1,01/01/2020,1,Alder #20,8.6,48,6795.467398,2
2,01/01/2020,2,Deadwood #23,10.4,53,16648.009634,1
3,01/01/2020,4,Trinity River #3,7.8,43,341.039988,2
4,01/01/2020,5,Sugar Pine #9,0.7,4,49178.633311,1
...,...,...,...,...,...,...,...
103510,09/01/2025,72,RJ Donovan Correctional Facility (RJD),13.6,59,970.920766,7
103511,09/01/2025,73,Otay Mesa Detention Center,13.6,59,498.496428,7
103512,09/01/2025,76,Calipatria State Prison (CAL),9.0,50,21129.294068,2
103513,09/01/2025,77,Centinela State Prison (CEN),10.7,54,21020.868292,3


### PurpleAir PM2.5 Data

In [110]:
# Utilizing PurpleAir to find missing facilities PM2.5 concentration
missing_ids_df = epa_single_fac[epa_single_fac['ID'].isin(missing_ids)]

# Manually found monitors on PurpleAir Map by copy-pasting facilities address'
# Note: first group of 3 correlate to Susanville addresses, second group of 2 to Blyth addresses, last two groups of 3 correlate to
# Redway and Bieber addresses
sensor_ids = [235169, 233187, 12302, 89643, 223283, 125293, 121137, 111419, 173785, 82983, 16625]

time_chunks = [
    ('2020-01-01T00:00:00Z', '2021-01-01T00:00:00Z'),
    ('2021-01-01T00:00:00Z', '2022-01-01T00:00:00Z'),
    ('2022-01-01T00:00:00Z', '2023-01-01T00:00:00Z'),
    ('2023-01-01T00:00:00Z', '2024-01-01T00:00:00Z'),
    ('2024-01-01T00:00:00Z', '2025-01-01T00:00:00Z'),
    ('2025-01-01T00:00:00Z', '2025-09-02T00:00:00Z')
]

# Request each monitor's data by iterating through sensor_ids
all_data_requested = []
for sensor in sensor_ids:
    for start, end in time_chunks: 
        purpleair_url = f"https://api.purpleair.com/v1/sensors/{sensor}/history/csv"
        headers = {"X-API-Key": PURPLEAIR_API_KEY}
        
        params = {
        'start_timestamp': start,
        'end_timestamp': end,
        'average': 1440,
        'fields': 'pm2.5_cf_1,humidity'
        }
        
        response = requests.get(purpleair_url, headers=headers, params=params)
        if response.status_code == 200:
            df = pd.read_csv(StringIO(response.text))
            df['sensor_index'] = sensor
            all_data_requested.append(df)
        else: 
            print(f"Failed to retrieve data with Response status code: {response.status_code}")
            
if all_data_requested:
    purpleair_sensors_df = pd.concat(all_data_requested)

purpleair_sensors_df

/var/folders/qf/cq7m_hx954l_s2pj_f4cs6nh0000gn/T/ipykernel_38379/4269654154.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  purpleair_sensors_df = pd.concat(all_data_requested)


,time_stamp,sensor_index,humidity,pm2.5_cf_1
0,2024-12-01T00:00:00Z,235169,57.0,18.6
1,2024-12-08T00:00:00Z,235169,53.0,25.1
2,2024-10-27T00:00:00Z,235169,27.0,0.7
3,2024-10-15T00:00:00Z,235169,27.0,5.8
4,2024-11-02T00:00:00Z,235169,51.0,4.9
...,...,...,...,...
239,2025-03-10T00:00:00Z,16625,34.0,3.5
240,2025-01-14T00:00:00Z,16625,23.0,1.9
241,2025-05-20T00:00:00Z,16625,27.0,2.2
242,2025-05-24T00:00:00Z,16625,33.0,5.7


In [163]:
# Correcting PurpleAir sensors to EPA standards:
# PAcf1 < 570 (corrected = 300 µg/m3 at 50% RH):
#     PM2.5 = PAcf1 × 0.524 − 0.0862 × RH + 5.75	(1)
# 570 ≤ PAcf1 < 611:
#     PM2.5 = (0.0244 × PAcf1 − 13.9) × [Equation (3)] + (1 − (0.0244 × PAcf1 − 13.9)) × [Equation (1)]	(2)
# PAcf1 ≥ 611 (corrected 400 µg/m3):
#     PM2.5 = PAcf12 × 4.21 × 10−4 + PAcf1 × 0.392 + 3.44

purpleair_sensors_df['Date'] = pd.to_datetime(purpleair_sensors_df['time_stamp'], format='ISO8601')
purpleair_sensors_df['Date'] = purpleair_sensors_df['Date'].dt.date

purpleair_sorted = purpleair_sensors_df.sort_values(['sensor_index', 'Date'])
purpleair_sorted['humidity'] = purpleair_sorted['humidity'].interpolate(method='linear')

def convert_purpleair_to_epa_standard(pm_cf, humidity):
    eq_1 = pm_cf * 0.524 - 0.0862 * humidity + 5.75
    eq_3 = pm_cf * 4.21 * ((10)**(-4)) + pm_cf * 0.392 + 3.44
    
    if pm_cf < 570:
        return eq_1

    elif 570 <= pm_cf < 611:
        weight = 0.0244 * pm_cf - 13.9
        return (weight) * (eq_3) + (1 - (weight)) * (eq_1)

    elif pm_cf >= 611:
        return eq_3
    
    return conversion
    
purpleair_sorted['epa_standard_conversion'] = list(map(convert_purpleair_to_epa_standard, purpleair_sorted['pm2.5_cf_1'], purpleair_sorted['humidity']))

purpleair_grouped = purpleair_sorted.groupby(['sensor_index', 'Date']).agg({
    'epa_standard_conversion':'max'
})
purpleair_grouped

epa_standard_conversion
sensor_index Date                               
12302        2021-08-12                  66.9600
             2021-08-13                  53.3986
             2021-08-14                 236.9626
             2021-08-15                  87.0394
             2021-08-16                 111.0014
...                                          ...
235169       2025-08-28                   6.5836
             2025-08-29                   6.0206
             2025-08-30                   7.0872
             2025-08-31                  13.8282
             2025-09-01                   7.5740

[9851 rows x 1 columns]

## Feature Engineering TROPOMI Data